# 项目数据分析：既有全量批次复现与五图

本 Notebook 是项目实施材料，不是课程原始 Notebook。本轮固定加载 **preset_deploy_20261020** 已完成的 CSV 和 manifest，复核统计并出图；**不重新读取数据库，不重跑 jieba、LDA 或 SnowNLP，不调用外部模型**。

来源是课程预置 chat.records(id,time,records)。原始 84,252 条，排除 NULL 文本 2,616 条，保留 81,636 条；这些历史文本尚不能确认属于热水器业务。已有模型计算完成，状态仍为 generated_pending_manual_review。

运行：在项目根目录或 notebooks 目录打开，选择 **Dify Project Python 3.10.9** 内核，从上至下执行。图像写入本批次 figures/，原 CSV、模型和 manifest 不会被改写。新数据或规则变更应另建分析批次；完整计算入口仍为 analysis/run_analysis.py。

In [ ]:
from datetime import datetime
from hashlib import sha256
import json
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import StrMethodFormatter
import numpy as np
import pandas as pd
from IPython.display import Image, display
from wordcloud import WordCloud

ROOT = Path.cwd()
if not (ROOT / "analysis" / "output").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "analysis" / "output").is_dir(), "请从项目根目录或 notebooks 目录运行。"
BATCH = "preset_deploy_20261020"
BATCH_DIR = ROOT / "analysis" / "output" / BATCH
FIGURE_DIR = BATCH_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
FONT_PATH = Path("C:/Windows/Fonts/msyh.ttc")
assert FONT_PATH.is_file(), "请将 FONT_PATH 指向本机可显示中文的字体。"
font_manager.fontManager.addfont(str(FONT_PATH))
plt.rcParams.update({
    "font.family": font_manager.FontProperties(fname=str(FONT_PATH)).get_name(),
    "axes.unicode_minus": False, "figure.dpi": 100, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
})
def file_hash(path):
    return sha256(Path(path).read_bytes()).hexdigest()
print({"python": sys.version.split()[0], "executable": sys.executable,
       "mode": "load_existing_verified_batch", "batch": BATCH})

## 1. 来源、哈希与已完成的清洗记录

五份 CSV 必须与同批次 manifest 的哈希、表头和行数完全一致。这里只展示已完成的清洗、分词与模型参数，不把加载缓存写成“本轮重新训练”。原快照和逐条映射保留在 data/private/analysis/preset_deploy_20261020/；本 Notebook 不读取或展示聊天正文、账号配置。

In [ ]:
manifest_path = BATCH_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert (manifest["source_mode"], manifest["batch_id"]) == ("preset", "deploy_20261020")
assert manifest["state"] == "generated_pending_manual_review"
reports = manifest["reports"]
tables, csv_checks = {}, []
for name, info in manifest["artifacts"].items():
    path = BATCH_DIR / name
    assert file_hash(path) == info["sha256"], f"文件哈希不一致：{name}"
    frame = pd.read_csv(path, encoding="utf-8-sig")
    assert list(frame.columns) == info["columns"] and len(frame) == info["rows"], name
    assert pd.api.types.is_integer_dtype(frame["count"]) and frame["count"].ge(0).all(), name
    tables[name] = frame
    csv_checks.append({"文件": name, "行数": len(frame), "表头": ", ".join(frame.columns), "SHA256核验": "一致"})
cleaning = reports["cleaning"]
assert cleaning["raw_rows"] == cleaning["retained_rows"] + cleaning["excluded_rows"]
display(pd.DataFrame(csv_checks))
display(pd.DataFrame([
    ["原始行数", cleaning["raw_rows"]], ["排除NULL文本", cleaning["excluded_null_text"]],
    ["排除空白文本", cleaning["excluded_blank_text"]], ["排除无效日期", cleaning["excluded_invalid_time"]],
    ["保留行数", cleaning["retained_rows"]], ["重复文本保留行数", cleaning["duplicate_text_rows_kept"]],
    ["分词总词次", reports["tokens"]["token_count"]], ["词典项数", reports["topics"]["dictionary_size"]],
], columns=["已有批次统计", "数值"]))
print({"time_min": cleaning["time_min"], "time_max": cleaning["time_max"],
       "lda_parameters": reports["topics"]["parameters"],
       "sentiment_thresholds": {"negative_below": reports["sentiment"]["negative_below"],
                                "positive_above": reports["sentiment"]["positive_above"]},
       "state": manifest["state"]})

## 2. 五图的分母与时间顺序

月份按年月连续排序；小时是整个数据期累计条数，不是单日平均。主题使用同一 LDA 模型的主主题编号 0/1/2，每条记录只计一次。词频是 token 出现次数 TOP100，不是聊天条数。

现有主题关键词高度重叠，保留编号，不擅自命名。SnowNLP 阈值 <0.4 负面、0.4≤score≤0.6 中性、>0.6 正面是项目暂定参数。**“负面”90.97% 是模型标签占比，不能直接解释为投诉率、不满意率或服务质量结论。**

In [ ]:
retained = int(cleaning["retained_rows"])
month = tables["聊天密度月分布.csv"].sort_values("month").reset_index(drop=True)
hour = tables["聊天密度小时分布.csv"].sort_values("hour").reset_index(drop=True)
words = tables["聊天词语频次数据.csv"]
topic_table = tables["月度主题趋势数据.csv"]
sentiment = tables["情感分布.csv"].set_index("sentiment").loc[["负面", "中性", "正面"]].reset_index()
trend = topic_table.pivot(index="month", columns="topic", values="count").sort_index().reindex(columns=[0, 1, 2])
assert month["month"].tolist() == pd.period_range(month["month"].iloc[0], month["month"].iloc[-1], freq="M").astype(str).tolist()
assert hour["hour"].tolist() == list(range(24))
assert month["count"].sum() == hour["count"].sum() == retained
assert len(topic_table) == len(month) * 3 and not topic_table.duplicated(["month", "topic"]).any()
assert not trend.isna().any().any() and set(topic_table["topic"]) == {0, 1, 2}
assert trend.index.tolist() == month["month"].tolist()
assert np.array_equal(trend.sum(axis=1).to_numpy(), month["count"].to_numpy())
assert topic_table["count"].sum() == reports["topics"]["assigned_rows"]
assert reports["topics"]["assigned_rows"] + reports["topics"]["unassigned_rows"] == retained
assert sentiment["count"].sum() == reports["sentiment"]["scored_rows"]
assert reports["sentiment"]["scored_rows"] + reports["sentiment"]["failed_rows"] == retained
assert len(words) == 100 and words["word"].is_unique and words["count"].is_monotonic_decreasing
word_total = int(words["count"].sum())
token_total = int(reports["tokens"]["token_count"])
assert word_total <= token_total
sentiment["占比"] = sentiment["count"] / reports["sentiment"]["scored_rows"]
display(sentiment.style.format({"count": "{:,}", "占比": "{:.2%}"}))
display(pd.DataFrame([
    {"主主题编号": int(topic), "条数": int(trend[topic].sum()),
     "实际TOP10关键词": "、".join(reports["topics"]["keywords"][str(topic)])}
    for topic in trend.columns
]))
print({"month_rows": len(month), "hour_rows": len(hour), "topic_rows": len(topic_table),
       "word_top100_count": word_total, "word_top100_share_of_tokens": round(word_total / token_total, 6),
       "topic_unassigned": reports["topics"]["unassigned_rows"],
       "sentiment_failed": reports["sentiment"]["failed_rows"], "checks": "passed"})

chart_files = []
def heading(fig, title, subtitle):
    fig.text(0.07, 0.95, title, fontsize=20, fontweight="bold", va="top", color="#17344a")
    fig.text(0.07, 0.89, subtitle, fontsize=11, va="top", color="#52616b")
def style_axes(ax):
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.18)
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.spines["left"].set_color("#bac5cc")
    ax.spines["bottom"].set_color("#bac5cc")
def finish(fig, name):
    fig.text(0.07, 0.075, f"来源：课程预置 chat.records，批次 {BATCH}，有效记录 {retained:,} 条。", fontsize=9, color="#52616b")
    fig.text(0.07, 0.035, "时间范围：2023-03-01 至 2024-08-28。模型与业务解释待人工复核，魔方导入尚未验收。", fontsize=9, color="#52616b")
    path = FIGURE_DIR / name
    fig.savefig(path, dpi=180, facecolor="white")
    chart_files.append(path)
    display(Image(filename=str(path)))
    plt.close(fig)

## 3. 月份面积图

18 个连续月份合计 81,636 条；2024 年 8 月只覆盖到 28 日。只描述数量，不据此确认增长原因，不把末月当作完整自然月。

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7.5))
fig.subplots_adjust(left=0.09, right=0.97, bottom=0.25, top=0.81)
heading(fig, "每月聊天记录量", "按 YYYY-MM 汇总的有效记录数；2024-08 仅覆盖至 8 月 28 日")
x, y = np.arange(len(month)), month["count"].to_numpy()
ax.fill_between(x, y, color="#1a9390", alpha=0.22)
ax.plot(x, y, color="#087f7b", linewidth=2.4, marker="o", markersize=4)
ax.set_xticks(x)
ax.set_xticklabels(month["month"], rotation=45, ha="right")
ax.set(xlabel="月份", ylabel="记录数（条）", xlim=(0, len(month) - 1), ylim=(0, max(y) * 1.24))
ax.annotate("末月覆盖至 8 月 28 日", xy=(x[-1], y[-1]), xytext=(x[-1] - 5, max(y) * 1.12),
            arrowprops={"arrowstyle": "->", "color": "#52616b"}, color="#52616b", fontsize=10)
style_axes(ax)
finish(fig, "01_月份面积图.png")

## 4. 小时柱状图

0–23 时为整个历史数据期的累计分布。峰值是 10 时的 5,943 条，不是“每天 10 时平均 5,943 条”。

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7.5))
fig.subplots_adjust(left=0.09, right=0.97, bottom=0.23, top=0.81)
heading(fig, "一天中各小时的聊天记录量", "整个数据期累计，非单日平均；0–23 时齐全")
peak = int(hour.loc[hour["count"].idxmax(), "hour"])
colors = ["#087f7b" if value == peak else "#7fb0cf" for value in hour["hour"]]
ax.bar(hour["hour"], hour["count"], color=colors, width=0.72)
ax.set_xticks(range(24))
ax.set_xticklabels([f"{value:02d}" for value in range(24)])
ax.set(xlabel="小时", ylabel="记录数（条）", ylim=(0, hour["count"].max() * 1.20))
peak_count = int(hour.loc[hour["hour"].eq(peak), "count"].iloc[0])
ax.text(peak, peak_count + 160, f"{peak} 时 {peak_count:,} 条", ha="center", fontsize=10, color="#087f7b")
style_axes(ax)
finish(fig, "02_小时柱状图.png")

## 5. TOP100 词云

直接使用已核验词频，不另删“吗、我、能、麻烦”等高频词，不调整停用词或重跑分词。TOP100 合计 615,128 词次，占全量 1,087,117 词次的 56.58%；高频功能词也说明当前词频的业务解释能力有限。

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7.5))
fig.subplots_adjust(left=0.05, right=0.98, bottom=0.18, top=0.82)
heading(fig, "聊天词频 TOP100", f"词频大小表示 token 出现次数；TOP100 占全部词次 {word_total / token_total:.2%}")
cloud = WordCloud(font_path=str(FONT_PATH), width=1800, height=950, background_color="white",
                  random_state=42, colormap="viridis", max_words=100, prefer_horizontal=0.9)
cloud.generate_from_frequencies(dict(zip(words["word"], words["count"])))
ax.imshow(cloud, interpolation="bilinear")
ax.axis("off")
finish(fig, "03_TOP100词云.png")

## 6. 月度主主题堆叠趋势

同一批次只使用一个 LDA 模型。K=3 为课程配置，未证明最优。关键词重叠较多，图例保留“主题 0/1/2”，不编造业务名称。每月堆叠总量等于对应月份记录量。

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7.5))
fig.subplots_adjust(left=0.09, right=0.97, bottom=0.25, top=0.80)
heading(fig, "月度主主题趋势", "同一 LDA 模型，K=3；每条记录只计一个主主题，主题尚未人工命名")
ax.stackplot(np.arange(len(trend)), *[trend[topic].to_numpy() for topic in trend.columns],
             labels=[f"主题 {topic}" for topic in trend.columns],
             colors=["#367fa9", "#dc9b45", "#51a293"], alpha=0.9)
ax.set_xticks(range(len(trend)))
ax.set_xticklabels(trend.index, rotation=45, ha="right")
ax.set(xlabel="月份", ylabel="主主题记录数（条）", xlim=(0, len(trend) - 1))
ax.legend(loc="upper left", frameon=False, ncol=3)
style_axes(ax)
finish(fig, "04_月度主题堆叠趋势.png")

## 7. 情感玫瑰图

SnowNLP 当前模型标签：负面 74,264 条、中性 2,837 条、正面 4,535 条，分母为成功评分的 81,636 条。等角扇区采用平方根半径，使**扇形面积与条数成正比**。90.97% 只是模型负面标签占比，不能写成投诉率或不满意率。

In [ ]:
fig = plt.figure(figsize=(12, 7.5))
heading(fig, "SnowNLP 情感标签分布", "等角扇区面积对应记录条数；半径为条数的平方根")
ax = fig.add_axes([0.05, 0.19, 0.55, 0.61], projection="polar")
theta = np.linspace(0, 2 * np.pi, 3, endpoint=False)
radii = np.sqrt(sentiment["count"].to_numpy())
palette = ["#bd6265", "#8b9ba8", "#3c9c95"]
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
ax.bar(theta, radii, width=2 * np.pi / 3 * 0.9, color=palette, alpha=0.88,
       edgecolor="white", linewidth=1.5)
ax.set_xticks(theta)
ax.set_xticklabels(sentiment["sentiment"], fontsize=12)
ticks = np.array([10000, 30000, 50000, 80000])
ax.set_yticks(np.sqrt(ticks))
ax.set_yticklabels([f"{value // 1000}千条" for value in ticks], fontsize=9, color="#52616b")
ax.set_ylim(0, np.sqrt(85000))
ax.set_rlabel_position(80)
ax.grid(alpha=0.22)
ax.spines["polar"].set_visible(False)
fig.text(0.65, 0.76, f"成功评分 {reports['sentiment']['scored_rows']:,} 条", fontsize=14, color="#17344a")
for position, row in sentiment.iterrows():
    fig.text(0.65, 0.67 - position * 0.09,
             f"{row['sentiment']}  {int(row['count']):,} 条  {row['占比']:.2%}",
             fontsize=13, color=palette[position])
fig.text(0.65, 0.34, "暂定阈值\nscore < 0.4：负面\n0.4 ≤ score ≤ 0.6：中性\nscore > 0.6：正面",
         fontsize=10, linespacing=1.7, color="#52616b")
fig.text(0.65, 0.19, "模型标签未人工校准\n不能直接代表投诉率或满意率", fontsize=10, color="#945256", linespacing=1.6)
finish(fig, "05_情感玫瑰图.png")

## 8. 保存图集证据与剩余验收

本轮只完成既有批次的加载、统计复核和可视化。T11 模型辅助清洗规则记录、异常样本人工复核、主题命名与 K 值评估、情感人工标注及阈值校准仍未完成。没有人工参照，不提供准确率或召回率。

CSV 表头是项目补充契约，尚未与真实课程魔方模板完成映射；五图是本地预览，不能据此宣称魔方大屏、CP5 或整个 P06/P07 已验收。原批次状态不变，原始聊天和凭据不进入输出。

In [ ]:
assert len(chart_files) == 5 and len(set(chart_files)) == 5
figure_manifest = {
    "status": "figures_generated_pending_manual_review",
    "generated_at": datetime.now().astimezone().isoformat(),
    "source_mode": manifest["source_mode"], "batch_directory": BATCH,
    "source_table": "chat.records", "source_manifest_sha256": file_hash(manifest_path),
    "source_batch_state": manifest["state"], "retained_rows": retained,
    "input_csv_sha256": {name: file_hash(BATCH_DIR / name) for name in tables},
    "figures": [{"path": str(path.relative_to(ROOT)), "sha256": file_hash(path), "bytes": path.stat().st_size}
                for path in chart_files],
    "model_training_executed_this_run": False, "sentiment_scoring_executed_this_run": False,
    "database_or_external_api_called": False, "manual_review_completed": False,
    "dashboard_import_verified": False, "pending_acceptance": manifest["pending_acceptance"],
    "chart_semantics": {
        "hour": "Whole-period cumulative counts, not daily averages",
        "month": "2023-03 to 2024-08; last month ends on 2024-08-28",
        "wordcloud": "Unmodified existing TOP100 token frequencies",
        "topic": "Dominant topic IDs 0/1/2 from one existing model; no business names assigned",
        "sentiment_rose": "Equal angular width; radius=sqrt(count), area proportional to count; model labels only",
    },
}
(FIGURE_DIR / "figure_manifest.json").write_text(json.dumps(figure_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([{"图像": path.name, "大小(字节)": path.stat().st_size} for path in chart_files]))
print({"figures": len(chart_files), "output": str(FIGURE_DIR.relative_to(ROOT)),
       "source_batch_state": manifest["state"], "manual_review": "pending", "dashboard_import": "pending"})